# JupyterLite で学ぶ pyvis 入門：ネットワークの対話的可視化

このノートブックは、**JupyterLite（ブラウザだけで動く Jupyter 環境）** 上で、
ネットワーク（グラフ）をマウスで動かせる形で描く Python ライブラリ **pyvis** の基礎を学ぶためのチュートリアルです。

## 対象者
- Python の基本（リスト、辞書、for 文）を理解している方
- NetworkX でネットワーク分析をしたことがある、または `python/networkx/networkx_beginner_tutorial.ipynb` を終えた方
- 貿易・企業間取引・SNS などのつながりを、見て分かる形で表現したい方

## このチュートリアルで学ぶこと
1. pyvis とは・JupyterLite での描画方法
2. ノードとエッジの属性（ラベル、ホバー表示、色、大きさ）
3. NetworkX のグラフからの変換と、中心性・コミュニティによる表現
4. レイアウトと物理エンジンの設定
5. 有向・重み付きネットワーク（貿易ネットワークの例）
6. HTML ファイルへの保存と、大きなグラフを扱うときの注意

## 使い方
- セルを上から順に `Shift + Enter` で実行してください。
- 描かれたネットワークは、**ドラッグで動かす・ホイールで拡大縮小・ノードにマウスを乗せると情報表示** ができます。
- 各章の最後に **練習問題** があります。「解答欄」に自分でコードを書いてから「解答例」を開いて確認しましょう。

## 0. 環境準備（JupyterLite 用）

まず、必要なライブラリをインストールします。pyvis は純 Python のパッケージなので PyPI から取得できます。

In [ ]:
# JupyterLite 用のパッケージインストール
try:
    import piplite
    await piplite.install(["scipy", "pyvis", "networkx", "numpy", "pandas"])
except ImportError:
    pass

import scipy  # networkx の一部の関数（レイアウト・中心性・PageRank）が内部で使うため先に読み込む


In [ ]:
import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import HTML, display
from pyvis.network import Network

print(f"NetworkX バージョン: {nx.__version__}")

---
## 1. pyvis とは・JupyterLite での描画方法

**pyvis** は、JavaScript のネットワーク描画ライブラリ **vis.js（vis-network）** を Python から使うための
ラッパーです。NetworkX で作ったグラフを、ドラッグやズームができる **対話的な図** として表示できます。

### JupyterLite で表示するときの約束

通常の Jupyter では `net.show("graph.html")` と書きますが、JupyterLite ではこの方法（iframe 表示）は
**画面に何も出ません**。代わりに、次の 2 点を守ります。

1. `Network(notebook=True, cdn_resources="remote")` で作る（描画用の JavaScript を CDN から読み込む設定）
2. `display(HTML(net.generate_html()))` で HTML をセルの出力に直接埋め込む

毎回書くのは面倒なので、関数 `show_network()` を用意しておきます。

In [ ]:
def show_network(net):
    """pyvis の Network をセルの出力に描画する（JupyterLite 対応）"""
    display(HTML(net.generate_html()))


def new_network(height="400px", directed=False):
    """JupyterLite 用の設定で Network を作る"""
    return Network(height=height, width="100%", notebook=True, cdn_resources="remote", directed=directed)


net = new_network()
net.add_node(1, label="A")
net.add_node(2, label="B")
net.add_node(3, label="C")
net.add_edge(1, 2)
net.add_edge(2, 3)
net.add_edge(3, 1)
show_network(net)

三角形のネットワークが表示されれば成功です。ノードをドラッグしてみてください（物理エンジンが働いて揺れ戻ります）。

---
## 2. ノードとエッジの属性

### 2.1 ノードの属性

`add_node(id, ...)` には次のような属性を渡せます。

| 引数 | 意味 |
|---|---|
| `label` | ノードの下（または中）に表示する文字 |
| `title` | マウスを乗せたときに表示する説明（ホバー） |
| `color` | 色（`"red"` や `"#1f77b4"`） |
| `size` | 大きさ（数値） |
| `value` | 値に応じて大きさを自動で決める（`size` の代わり） |
| `shape` | 形（`"dot"`, `"box"`, `"circle"`, `"star"`, `"triangle"` など） |

In [ ]:
net = new_network()
net.add_node("東京", label="東京", title="人口 約1400万人", color="#e74c3c", size=40)
net.add_node("名古屋", label="名古屋", title="人口 約230万人", color="#3498db", size=25)
net.add_node("大阪", label="大阪", title="人口 約280万人", color="#2ecc71", size=28)
net.add_node("福岡", label="福岡", title="人口 約160万人", color="#f39c12", size=22, shape="box")
net.add_edge("東京", "名古屋")
net.add_edge("名古屋", "大阪")
net.add_edge("大阪", "福岡")
net.add_edge("東京", "大阪")
show_network(net)

### 2.2 エッジの属性

`add_edge(a, b, ...)` の主な属性です。

| 引数 | 意味 |
|---|---|
| `title` | ホバーで表示する説明 |
| `value` | 値に応じて線の太さを自動で決める |
| `width` | 線の太さを直接指定 |
| `color` | 線の色 |
| `dashes` | `True` で点線 |

In [ ]:
net = new_network()
for city in ["東京", "名古屋", "大阪", "福岡"]:
    net.add_node(city, label=city, size=20)
net.add_edge("東京", "名古屋", value=10, title="新幹線 約1時間40分")
net.add_edge("名古屋", "大阪", value=5, title="新幹線 約50分", color="#2ecc71")
net.add_edge("大阪", "福岡", value=3, title="新幹線 約2時間30分", dashes=True)
net.add_edge("東京", "大阪", value=15, title="新幹線 約2時間30分", color="#e74c3c")
show_network(net)

### 2.3 まとめて追加する：add_nodes / add_edges

ノードやエッジが多いときは、リストでまとめて追加できます。`add_nodes` では属性もリストで渡します。

In [ ]:
net = new_network()
ids = list(range(1, 7))
labels = [f"企業{i}" for i in ids]
sizes = [10, 15, 20, 25, 30, 35]
colors = ["#1f77b4"] * 3 + ["#ff7f0e"] * 3
net.add_nodes(ids, label=labels, size=sizes, color=colors, title=[f"従業員 {s * 10} 人" for s in sizes])
net.add_edges([(1, 2), (1, 3), (2, 4), (3, 5), (4, 6), (5, 6), (2, 5)])
show_network(net)
print("ノード数:", len(net.nodes), " エッジ数:", len(net.edges))

### 練習問題 1

1. 自分と友人 4 人（合計 5 ノード）のネットワークを作り、自分のノードだけ赤くて大きく（`size=30`）、友人同士のつながりも 2 本以上入れて表示してください。
2. 各ノードの `title` に「趣味」を入れ、エッジの `value` に「親しさ（1〜5）」を入れて、線の太さが変わることを確認してください。

In [ ]:
# 練習問題 1 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 1 の解答例を見る</strong></summary>

```python
net = new_network()
net.add_node("自分", label="自分", color="red", size=30, title="趣味: 読書")
friends = {"友人A": "サッカー", "友人B": "料理", "友人C": "旅行", "友人D": "ゲーム"}
for name, hobby in friends.items():
    net.add_node(name, label=name, size=15, title=f"趣味: {hobby}")
net.add_edge("自分", "友人A", value=5)
net.add_edge("自分", "友人B", value=3)
net.add_edge("自分", "友人C", value=2)
net.add_edge("自分", "友人D", value=4)
net.add_edge("友人A", "友人B", value=1)
net.add_edge("友人C", "友人D", value=3)
show_network(net)
```

</details>

---
## 3. NetworkX のグラフから変換する

分析は NetworkX、表示は pyvis、という組み合わせが基本です。
`net.from_nx(G)` で NetworkX のグラフを取り込めます。NetworkX 側でノードに付けた属性
（`size`, `color`, `title`, `label` など）は、そのまま pyvis の属性として使われます。

### 3.1 from_nx の基本

In [ ]:
G = nx.karate_club_graph()          # 空手クラブの友人関係（34 人）
print("ノード数:", G.number_of_nodes(), " エッジ数:", G.number_of_edges())

net = new_network(height="500px")
net.from_nx(G)
show_network(net)

### 3.2 中心性でノードの大きさを変える

「誰が重要か」を **中心性** で測り、ノードの大きさに反映させます。
NetworkX で属性を付けてから `from_nx` します。

In [ ]:
degree = nx.degree_centrality(G)
betweenness = nx.betweenness_centrality(G)

for n in G.nodes:
    G.nodes[n]["size"] = 10 + 60 * degree[n]                       # 次数中心性 → 大きさ
    G.nodes[n]["title"] = f"ノード {n}<br>次数中心性 {degree[n]:.2f}<br>媒介中心性 {betweenness[n]:.2f}"
    G.nodes[n]["label"] = str(n)

net = new_network(height="500px")
net.from_nx(G)
show_network(net)

top = sorted(degree, key=degree.get, reverse=True)[:3]
print("次数中心性の上位 3 ノード:", top)

### 3.3 コミュニティで色分けする

`greedy_modularity_communities` でコミュニティ（つながりの密なグループ）を検出し、グループごとに色を塗り分けます。

In [ ]:
from networkx.algorithms import community

communities = community.greedy_modularity_communities(G)
palette = ["#e41a1c", "#377eb8", "#4daf4a", "#984ea3", "#ff7f00", "#a65628"]
for i, members in enumerate(communities):
    for n in members:
        G.nodes[n]["color"] = palette[i % len(palette)]
        G.nodes[n]["group"] = i

net = new_network(height="500px")
net.from_nx(G)
show_network(net)
print("コミュニティ数:", len(communities), " 各サイズ:", [len(c) for c in communities])

### 練習問題 2

1. `nx.les_miserables_graph()`（レ・ミゼラブルの登場人物の共演関係）を読み込み、媒介中心性でノードの大きさを決めて表示してください（`title` に名前と媒介中心性を入れます）。
2. 同じグラフをコミュニティで色分けし、コミュニティ数を表示してください。

In [ ]:
# 練習問題 2 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 2 の解答例を見る</strong></summary>

```python
LM = nx.les_miserables_graph()
bc = nx.betweenness_centrality(LM)
for n in LM.nodes:
    LM.nodes[n]["size"] = 8 + 80 * bc[n]
    LM.nodes[n]["title"] = f"{n}<br>媒介中心性 {bc[n]:.3f}"

comms = community.greedy_modularity_communities(LM)
palette = ["#e41a1c", "#377eb8", "#4daf4a", "#984ea3", "#ff7f00", "#a65628", "#f781bf", "#999999"]
for i, members in enumerate(comms):
    for n in members:
        LM.nodes[n]["color"] = palette[i % len(palette)]

net = new_network(height="550px")
net.from_nx(LM)
show_network(net)
print("コミュニティ数:", len(comms))
```

</details>

---
## 4. レイアウトと物理エンジン

pyvis は既定で **物理エンジン** を使い、ノード同士が反発・引力で自然な配置に落ち着きます。
ノードが多いと動きが重くなるので、状況に応じて設定を変えます。

### 4.1 物理エンジンを止めて座標を固定する

NetworkX の `spring_layout` で座標を計算し、`x`, `y` 属性として渡してから `toggle_physics(False)` にすると、
配置が固定された軽い図になります（座標は数百ピクセル程度に拡大します）。

In [ ]:
pos = nx.spring_layout(G, seed=1)
for n, (x, y) in pos.items():
    G.nodes[n]["x"] = float(x * 400)
    G.nodes[n]["y"] = float(y * 400)

net = new_network(height="500px")
net.from_nx(G)
net.toggle_physics(False)        # 物理エンジンを止める（ノードは動かせるが戻らない）
show_network(net)

### 4.2 物理エンジンのパラメータを調整する

`barnes_hut()` や `repulsion()` で、引力・反発力・ばねの長さなどを変えられます。

In [ ]:
net = new_network(height="500px")
net.from_nx(G)
net.barnes_hut(gravity=-4000, central_gravity=0.3, spring_length=120, spring_strength=0.02)
show_network(net)

### 4.3 set_options で細かく設定する

vis.js の設定を JSON 形式の文字列で直接渡せます。よく使うのは、ノードの形・エッジの滑らかさ・物理エンジンの安定化です。

> `net.show_buttons()` を使うと設定パネルを表示できますが、JupyterLite では画面が重くなるので、このノートでは使いません。

In [ ]:
net = new_network(height="500px")
net.from_nx(G)
net.set_options("""
{
  "nodes": {"shape": "dot", "font": {"size": 14}},
  "edges": {"color": {"color": "#bbbbbb"}, "smooth": false},
  "physics": {"stabilization": {"iterations": 200}, "barnesHut": {"gravitationalConstant": -3000}}
}
""")
show_network(net)

### 練習問題 3

1. `nx.circular_layout(G)` の座標を使って、空手クラブのグラフを円形に固定配置で表示してください。
2. `set_options` で、エッジを点線（`"edges": {"dashes": true}`）、ノードの形を `"box"` にして表示してください。

In [ ]:
# 練習問題 3 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 3 の解答例を見る</strong></summary>

```python
# 1
pos = nx.circular_layout(G)
for n, (x, y) in pos.items():
    G.nodes[n]["x"] = float(x * 300)
    G.nodes[n]["y"] = float(y * 300)
net = new_network(height="500px")
net.from_nx(G)
net.toggle_physics(False)
show_network(net)

# 2
net = new_network(height="500px")
net.from_nx(G)
net.set_options('{"edges": {"dashes": true}, "nodes": {"shape": "box"}}')
show_network(net)
```

</details>

---
## 5. 有向・重み付きネットワーク：貿易ネットワークの例

国と国の輸出額のような **向きと重みのあるつながり** を描きます。ここでは仮のデータを乱数で作ります。

### 5.1 仮の貿易データを作る

In [ ]:
rng = np.random.default_rng(0)
countries = ["日本", "中国", "米国", "韓国", "ドイツ", "タイ", "ベトナム", "豪州"]
rows = []
for exp in countries:
    for imp in countries:
        if exp != imp and rng.random() < 0.6:            # 60% の組み合わせに輸出がある
            rows.append({"輸出国": exp, "輸入国": imp, "輸出額": int(rng.lognormal(3, 1) * 10)})
trade = pd.DataFrame(rows)
print(trade.head(10))
print("エッジ数:", len(trade))

### 5.2 有向グラフとして描く

`new_network(directed=True)` で矢印付きになります。`value` に輸出額を渡すと太さに、`title` に金額を入れるとホバーで確認できます。

In [ ]:
net = new_network(height="500px", directed=True)
for c in countries:
    net.add_node(c, label=c, size=20)
for _, r in trade.iterrows():
    net.add_edge(r["輸出国"], r["輸入国"], value=int(r["輸出額"]), title=f"{r['輸出国']}→{r['輸入国']}: {r['輸出額']} 億ドル")
show_network(net)

### 5.3 輸出総額でノードの大きさを決め、小さな取引は省く

エッジが多いと見づらいので、**閾値** 以上の取引だけを描きます。ノードの大きさは輸出総額にします。

In [ ]:
export_total = trade.groupby("輸出国")["輸出額"].sum()
threshold = trade["輸出額"].quantile(0.5)          # 中央値以上だけ表示
big_trade = trade[trade["輸出額"] >= threshold]

net = new_network(height="500px", directed=True)
for c in countries:
    total = int(export_total.get(c, 0))
    net.add_node(c, label=c, size=10 + total / 20, title=f"{c}<br>輸出総額 {total} 億ドル")
for _, r in big_trade.iterrows():
    net.add_edge(r["輸出国"], r["輸入国"], value=int(r["輸出額"]), title=f"{r['輸出額']} 億ドル")
show_network(net)
print("表示したエッジ数:", len(big_trade), "/", len(trade))

### 5.4 NetworkX で分析してから描く

pandas の表から `nx.from_pandas_edgelist` で有向グラフを作り、**入次数・出次数（重み付き）** を計算して、
輸入超過の国と輸出超過の国を色で区別します。

In [ ]:
DG = nx.from_pandas_edgelist(trade, "輸出国", "輸入国", edge_attr="輸出額", create_using=nx.DiGraph)
out_strength = dict(DG.out_degree(weight="輸出額"))     # 輸出総額
in_strength = dict(DG.in_degree(weight="輸出額"))       # 輸入総額

for n in DG.nodes:
    balance = out_strength[n] - in_strength[n]
    DG.nodes[n]["size"] = 10 + (out_strength[n] + in_strength[n]) / 40
    DG.nodes[n]["color"] = "#2ecc71" if balance >= 0 else "#e74c3c"
    DG.nodes[n]["title"] = f"{n}<br>輸出 {out_strength[n]} / 輸入 {in_strength[n]}<br>収支 {balance:+}"
    DG.nodes[n]["label"] = n
for u, v, d in DG.edges(data=True):
    d["value"] = d["輸出額"]
    d["title"] = f"{u}→{v}: {d['輸出額']} 億ドル"

net = new_network(height="500px", directed=True)
net.from_nx(DG)
show_network(net)

summary = pd.DataFrame({"輸出": out_strength, "輸入": in_strength})
summary["収支"] = summary["輸出"] - summary["輸入"]
print(summary.sort_values("収支", ascending=False))

### 練習問題 4

1. `trade` から「日本が関わる取引（輸出国または輸入国が日本）」だけを取り出し、有向グラフで表示してください。
2. 輸入総額（`in_strength`）が最も大きい国を求め、その国を星形（`shape="star"`）で表示してください。

In [ ]:
# 練習問題 4 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 4 の解答例を見る</strong></summary>

```python
# 1
jp = trade[(trade["輸出国"] == "日本") | (trade["輸入国"] == "日本")]
net = new_network(height="450px", directed=True)
for c in set(jp["輸出国"]) | set(jp["輸入国"]):
    net.add_node(c, label=c, size=20, color="#e74c3c" if c == "日本" else "#3498db")
for _, r in jp.iterrows():
    net.add_edge(r["輸出国"], r["輸入国"], value=int(r["輸出額"]), title=f"{r['輸出額']} 億ドル")
show_network(net)

# 2
top_importer = max(in_strength, key=in_strength.get)
print("輸入総額が最大の国:", top_importer, in_strength[top_importer])
for n in DG.nodes:
    DG.nodes[n]["shape"] = "star" if n == top_importer else "dot"
net = new_network(height="450px", directed=True)
net.from_nx(DG)
show_network(net)
```

</details>

---
## 6. HTML ファイルへの保存と大きなグラフの注意

### 6.1 HTML として保存する

`net.save_graph("ファイル名.html")` で、単体で開ける HTML ファイルになります。
JupyterLite では左のファイルブラウザに現れるので、右クリック → Download で手元に保存し、ブラウザで開けます。

In [ ]:
import os

net = new_network(height="500px", directed=True)
net.from_nx(DG)
net.save_graph("trade_network.html")
print("保存しました:", os.path.exists("trade_network.html"), " サイズ:", os.path.getsize("trade_network.html"), "バイト")

### 6.2 大きなグラフを扱うときの注意

ノードが数百を超えると、ブラウザ上の物理エンジンの計算が重くなります。対策は次のとおりです。

1. **サブグラフに絞る**：次数の大きい上位ノードだけ、`k_core` で密な部分だけ、など
2. **物理エンジンを止める**：座標を NetworkX で計算して固定する（4.1 節）
3. **エッジを間引く**：重みが小さいエッジを閾値で省く（5.3 節）

例として、300 ノードのランダムグラフから次数上位 40 ノードのサブグラフだけを描きます。

In [ ]:
big = nx.barabasi_albert_graph(300, 2, seed=3)
print("元のグラフ:", big.number_of_nodes(), "ノード", big.number_of_edges(), "エッジ")

top40 = sorted(big.degree, key=lambda x: x[1], reverse=True)[:40]
sub = big.subgraph([n for n, _ in top40]).copy()
for n in sub.nodes:
    sub.nodes[n]["size"] = 5 + big.degree[n]
    sub.nodes[n]["title"] = f"ノード {n}（次数 {big.degree[n]}）"
    sub.nodes[n]["label"] = str(n)
print("サブグラフ:", sub.number_of_nodes(), "ノード", sub.number_of_edges(), "エッジ")

pos = nx.spring_layout(sub, seed=1)
for n, (x, y) in pos.items():
    sub.nodes[n]["x"] = float(x * 400)
    sub.nodes[n]["y"] = float(y * 400)
net = new_network(height="500px")
net.from_nx(sub)
net.toggle_physics(False)
show_network(net)

---
## まとめ

| トピック | 主な関数・設定 |
|---|---|
| JupyterLite での描画 | `Network(notebook=True, cdn_resources="remote")` → `display(HTML(net.generate_html()))` |
| ノード | `add_node(id, label, title, color, size, value, shape)`, `add_nodes([...])` |
| エッジ | `add_edge(a, b, title, value, width, color, dashes)`, `add_edges([...])` |
| NetworkX 連携 | `net.from_nx(G)`（ノード属性 `size`/`color`/`title`/`x`/`y` がそのまま使われる） |
| 分析との組み合わせ | `degree_centrality`, `betweenness_centrality`, `greedy_modularity_communities`, `from_pandas_edgelist` |
| レイアウト | `toggle_physics(False)` + `spring_layout` の座標, `barnes_hut()`, `set_options(JSON)` |
| 保存 | `net.save_graph("file.html")` |

## 次のステップ

- `python/networkx/networkx_beginner_tutorial.ipynb` — 中心性やコミュニティ検出など、分析手法をより詳しく
- `python/mesa/mesa_beginner_tutorial.ipynb` — ネットワーク上の情報伝播をシミュレーションする
- 実データ（企業の取引関係、共著関係、SNS のフォロー関係など）を CSV で用意し、`from_pandas_edgelist` から同じ手順で描いてみましょう

---
## 総合演習：企業間取引ネットワークの可視化

次の手順で、仮想の企業間取引ネットワークを分析・可視化してください。

1. 企業 30 社（`企業01`〜`企業30`）の取引関係を乱数で作る。各企業は 2〜4 社に対して取引額（10〜100）を持つ有向エッジを持つ（`np.random.default_rng(1)` を使う）。
2. NetworkX の有向グラフにし、**取引額で重み付けした入次数**（受注額）を計算する。
3. コミュニティ検出は無向グラフに変換（`DG.to_undirected()`）してから行い、コミュニティごとに色を付ける。
4. ノードの大きさ＝受注額、`title` に受注額と発注額を入れ、エッジの太さ＝取引額として有向グラフで表示する。
5. 受注額の上位 5 社を表で表示し、図を `firm_network.html` に保存する。

In [ ]:
# 総合演習の解答欄：ここにコードを書いてください

### 総合演習の解答例

自分で書いてから、次のセルを実行して結果を比べてみてください。

In [ ]:
rng = np.random.default_rng(1)
firms = [f"企業{i:02d}" for i in range(1, 31)]
edges = []
for f in firms:
    partners = rng.choice([x for x in firms if x != f], size=rng.integers(2, 5), replace=False)
    for p in partners:
        edges.append({"発注": f, "受注": p, "取引額": int(rng.integers(10, 101))})
deals = pd.DataFrame(edges)

DG2 = nx.from_pandas_edgelist(deals, "発注", "受注", edge_attr="取引額", create_using=nx.DiGraph)
received = dict(DG2.in_degree(weight="取引額"))    # 受注額
placed = dict(DG2.out_degree(weight="取引額"))     # 発注額

comms = community.greedy_modularity_communities(DG2.to_undirected())
palette = ["#e41a1c", "#377eb8", "#4daf4a", "#984ea3", "#ff7f00", "#a65628", "#f781bf"]
for i, members in enumerate(comms):
    for n in members:
        DG2.nodes[n]["color"] = palette[i % len(palette)]
for n in DG2.nodes:
    DG2.nodes[n]["size"] = 8 + received[n] / 15
    DG2.nodes[n]["title"] = f"{n}<br>受注額 {received[n]} / 発注額 {placed[n]}"
    DG2.nodes[n]["label"] = n
for u, v, d in DG2.edges(data=True):
    d["value"] = d["取引額"]
    d["title"] = f"{u}→{v}: {d['取引額']}"

net = new_network(height="550px", directed=True)
net.from_nx(DG2)
show_network(net)

top5 = pd.Series(received, name="受注額").sort_values(ascending=False).head(5)
print(top5)
net.save_graph("firm_network.html")
print("コミュニティ数:", len(comms), " 保存:", os.path.exists("firm_network.html"))

お疲れさまでした！ pyvis を使うと、ネットワーク分析の結果を「動かして確かめられる図」として共有できます。
分析は NetworkX、見せ方は pyvis、と役割分担して使ってみてください。